[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C30_Agent_Harness_Course/01_agent_loop/01_agent_loop.ipynb)

# 01 · Agent 循环（用 MockLLM 端到端跑通）

目标：亲手把 **agent 循环**从零写对——**感知→决策→行动→观察**、**ReAct 文本解析**、**停止条件**、**max-steps 守卫**、**状态管理**，全部用确定性 **MockLLM** 端到端运行、用 `assert` 验证 agent 确实多步完成了任务。

路线：MockLLM 与动作协议 → 最小循环(四阶段) → ReAct 文本解析 → 停止条件状态机 → 状态管理(history) → 完整循环 → ✏️ 练习 → 📖 答案 → 🧪 真实 Claude 适配胶囊。

> 心智模型：**循环是 while；记忆是不断变长的 history；决策是唯一调模型处；其余全是普通程序逻辑**。纯标准库，无需 API key。

## 1 · MockLLM 与动作协议

先定义 agent 循环与「大脑」之间的契约：大脑的 `complete(history, tools)` 返回一个**结构化决策**。
MockLLM 按预设脚本返回，确定性、零成本，让我们能 `assert` 每一步。动作两类：调工具 / 给最终答案。

In [ ]:
import json

class MockLLM:
    '''假大脑：按脚本逐步返回结构化决策。每条脚本是一个 dict：
         {'stop_reason':'tool_use', 'tool_calls':[{'id','name','input'}], 'text':...}
         {'stop_reason':'end_turn', 'text': '最终答案'}
       结构刻意与 Claude Messages API 同构(stop_reason / tool_use)。'''
    def __init__(self, script):
        self.script = list(script)
        self.calls = 0
    def complete(self, history, tools=None):
        assert self.calls < len(self.script), 'MockLLM 脚本用尽(可能 agent 没按预期停止)'
        resp = self.script[self.calls]
        self.calls += 1
        return resp

# 一个两步脚本：调 search，再据结果作答
brain = MockLLM(script=[
    {'stop_reason':'tool_use', 'text':'我先搜一下',
     'tool_calls':[{'id':'t1','name':'search','input':{'q':'agent'}}]},
    {'stop_reason':'end_turn', 'text':'agent 是模型+循环+工具+上下文。'},
])
r0 = brain.complete([])
print('第1次决策 stop_reason =', r0['stop_reason'], '| 调用工具:', r0['tool_calls'][0]['name'])
assert r0['stop_reason'] == 'tool_use'
assert brain.calls == 1
print('✅ MockLLM 契约就绪：complete() 返回结构化决策，确定性可断言')

## 2 · 最小 agent 循环：四阶段跑通一个多步任务

把**感知→决策→行动→观察**写成一个 `while`。这次结构完整：维护 history、追加模型决策、分发工具、回灌观察、正常停止、max-steps 守卫。

In [ ]:
def dispatch(tools, call):
    '''执行一个工具调用，返回 observation(带 tool_use_id 便于对位)。'''
    fn = tools[call['name']]
    out = fn(**call['input'])
    return {'tool_use_id': call['id'], 'content': str(out)}

# 上一格已经消耗了 brain 一次, 这里新建一个全新的 brain 跑完整循环
brain = MockLLM(script=[
    {'stop_reason':'tool_use', 'text':'我先搜一下',
     'tool_calls':[{'id':'t1','name':'search','input':{'q':'agent'}}]},
    {'stop_reason':'end_turn', 'text':'agent 是模型+循环+工具+上下文。'},
])

def run_agent(llm, tools, task, max_steps=10):
    history = [{'role':'user', 'content':task}]   # 状态/scratchpad
    trace = []
    for step in range(max_steps):                 # 守卫：防死循环
        resp = llm.complete(history, tools)        # ②决策
        history.append({'role':'assistant', 'content':resp.get('text',''),
                        'tool_calls':resp.get('tool_calls')})  # 状态:追加决策
        if resp['stop_reason'] == 'end_turn':       # ③正常停止
            trace.append(('final', resp['text']))
            return {'status':'done', 'answer':resp['text'], 'trace':trace, 'steps':step+1}
        obs = []
        for call in resp['tool_calls']:            # ③行动:分发
            o = dispatch(tools, call)
            obs.append(o)
            trace.append(('tool', call['name'], o['content']))
        history.append({'role':'user', 'content':obs})  # ④观察回灌
    return {'status':'max_steps', 'trace':trace, 'steps':max_steps}

TOOLS = {'search': lambda q: f'关于 {q}: 它是模型+循环+工具+上下文'}
out = run_agent(brain, TOOLS, 'agent 是什么?')
print('status:', out['status'], '| steps:', out['steps'])
print('answer:', out['answer'])
for t in out['trace']: print('  ', t)
assert out['status'] == 'done'
assert out['trace'][0][0] == 'tool' and out['trace'][0][1] == 'search'
assert out['trace'][1][0] == 'final'
assert out['steps'] == 2, '应 2 步完成'
print('✅ 四阶段循环端到端跑通：决策->分发->回灌->正常停止，2 步完成')

## 3 · ReAct 文本解析：把模型的文本变成动作

真实(不支持原生工具调用的)模型输出**文本**。最经典格式是 ReAct：`Thought/Action/Action Input/Final Answer`。
我们写一个解析器把它变成结构化动作——这是循环里最易出 bug 的一步。

In [ ]:
def parse_react(text):
    '''解析一段 ReAct 文本 -> 结构化动作。
       Final Answer 优先；否则抓 Action / Action Input(JSON)。'''
    if 'Final Answer:' in text:
        return {'stop_reason':'end_turn', 'text': text.split('Final Answer:')[1].strip()}
    if 'Action:' not in text:
        raise ValueError('既无 Final Answer 也无 Action，无法解析')
    action = text.split('Action:')[1].split('\n')[0].strip()
    raw = text.split('Action Input:')[1].strip()
    args = json.loads(raw)
    return {'stop_reason':'tool_use',
            'tool_calls':[{'id':'t', 'name':action, 'input':args}]}

txt1 = 'Thought: 我得查天气\nAction: get_weather\nAction Input: {"city": "北京"}'
txt2 = 'Thought: 够了\nFinal Answer: 北京晴 22°C'
a1 = parse_react(txt1)
a2 = parse_react(txt2)
print('解析工具调用:', a1['tool_calls'][0])
print('解析最终答案:', a2['text'])
assert a1['stop_reason'] == 'tool_use' and a1['tool_calls'][0]['name'] == 'get_weather'
assert a1['tool_calls'][0]['input'] == {'city':'北京'}
assert a2['stop_reason'] == 'end_turn' and '北京' in a2['text']
print('✅ ReAct 解析器正确：能区分工具调用与最终答案、能抽出工具名与 JSON 参数')

## 4 · 停止条件：一个明确的终止状态机

agent 退出循环时必须带**明确的终止原因**：done / max_steps / budget_exceeded。
给循环加上预算守卫，并验证每种停止都能被正确识别。

In [ ]:
def run_agent_v2(llm, tools, task, max_steps=10, max_cost=1.0):
    history = [{'role':'user','content':task}]
    cost = 0.0
    for step in range(max_steps):
        resp = llm.complete(history, tools)
        cost += resp.get('cost', 0.0)
        if cost > max_cost:
            return {'status':'budget_exceeded', 'cost':cost, 'steps':step}
        history.append({'role':'assistant','content':resp.get('text','')})
        if resp['stop_reason'] == 'end_turn':
            return {'status':'done', 'answer':resp['text'], 'steps':step+1}
        for call in resp['tool_calls']:
            o = dispatch(tools, call)
            history.append({'role':'user','content':[o]})
    return {'status':'max_steps', 'steps':max_steps}

# A) 正常完成
ok = run_agent_v2(MockLLM([{'stop_reason':'end_turn','text':'done'}]), {}, 'x')
# B) 超步数：永远调工具
loop_script = [{'stop_reason':'tool_use','text':'',
                'tool_calls':[{'id':'t','name':'search','input':{'q':'x'}}]}] * 50
over = run_agent_v2(MockLLM(loop_script), TOOLS, 'x', max_steps=4)
# C) 超预算：每步标价 0.6
costly = [{'stop_reason':'tool_use','text':'','cost':0.6,
           'tool_calls':[{'id':'t','name':'search','input':{'q':'x'}}]}] * 50
broke = run_agent_v2(MockLLM(costly), TOOLS, 'x', max_steps=10, max_cost=1.0)
print('A:', ok['status'], '| B:', over['status'], '| C:', broke['status'])
assert ok['status'] == 'done'
assert over['status'] == 'max_steps' and over['steps'] == 4
assert broke['status'] == 'budget_exceeded'
print('✅ 终止状态机正确：done / max_steps / budget_exceeded 都能被识别并带原因')

## 5 · 状态管理：忘了追加决策 = 死循环

agent 的记忆全在 history 里。**最经典的 bug：忘了把模型决策追加回 history**，导致模型每轮「失忆」、反复做同一决定。
我们用一个「会失忆的循环」对照「正常循环」，看清回灌的重要性。

In [ ]:
def history_len_after(llm, tools, task, append_decision, max_steps=5):
    '''append_decision=False 模拟『忘了追加模型决策』的 bug。返回最终 history。'''
    history = [{'role':'user','content':task}]
    for step in range(max_steps):
        resp = llm.complete(history, tools)
        if append_decision:
            history.append({'role':'assistant','content':resp.get('text','')})
        if resp['stop_reason'] == 'end_turn':
            return history
        for call in resp['tool_calls']:
            history.append({'role':'user','content':[dispatch(tools, call)]})
    return history

script = [
    {'stop_reason':'tool_use','text':'查','tool_calls':[{'id':'t','name':'search','input':{'q':'x'}}]},
    {'stop_reason':'end_turn','text':'答完'},
]
good = history_len_after(MockLLM(script), TOOLS, 'x', append_decision=True)
bad  = history_len_after(MockLLM(script), TOOLS, 'x', append_decision=False)
# 正常：user(task) + assistant(决策) + user(观察) + assistant(答) = 4
# 失忆：user(task) + user(观察) = 缺了两条 assistant
print('正常 history 长度:', len(good), '| 失忆 history 长度:', len(bad))
roles_good = [m['role'] for m in good]
assert 'assistant' in roles_good, '正常循环应含 assistant 决策'
assert len(good) > len(bad), '失忆 bug 会丢失模型决策、history 更短'
assert [m['role'] for m in bad].count('assistant') == 0, '失忆版本完全没记住模型说过什么'
print('✅ 看清了：不把模型决策追加回 history，模型就会失忆 -> 真实里这正是死循环的根源')

## 6 · 完整循环：带可读轨迹的 agent

把四阶段、停止状态机、状态管理合到一起，并产出一份**可读轨迹**——这是调试 agent 的利器。跑一个 3 步任务(搜索→计算→作答)。

In [ ]:
def run_agent_full(llm, tools, task, max_steps=10):
    history = [{'role':'user','content':task}]
    trace = []
    for step in range(max_steps):
        resp = llm.complete(history, tools)
        if resp.get('text'):
            trace.append(f'[step {step}] THOUGHT: {resp["text"]}')
        history.append({'role':'assistant','content':resp.get('text',''),
                        'tool_calls':resp.get('tool_calls')})
        if resp['stop_reason'] == 'end_turn':
            trace.append(f'[step {step}] FINAL: {resp["text"]}')
            return {'status':'done','answer':resp['text'],'trace':trace,'steps':step+1}
        for call in resp['tool_calls']:
            o = dispatch(tools, call)
            trace.append(f'[step {step}] ACTION {call["name"]}({call["input"]}) -> {o["content"]}')
            history.append({'role':'user','content':[o]})
    return {'status':'max_steps','trace':trace,'steps':max_steps}

tools = {
    'search': lambda q: '半径 7',
    'calc':   lambda expr: round(eval(expr, {'__builtins__':{}}, {}), 2),
}
brain3 = MockLLM([
    {'stop_reason':'tool_use','text':'先查半径','tool_calls':[{'id':'1','name':'search','input':{'q':'圆半径'}}]},
    {'stop_reason':'tool_use','text':'算面积','tool_calls':[{'id':'2','name':'calc','input':{'expr':'3.14159*7*7'}}]},
    {'stop_reason':'end_turn','text':'面积约 153.94'},
])
out = run_agent_full(brain3, tools, '半径为搜索结果的圆面积?')
print('\n'.join(out['trace']))
assert out['status'] == 'done' and out['steps'] == 3
assert '153.9' in out['answer']
print('\n✅ 完整循环跑通 3 步任务(搜索->计算->作答)，并产出可读轨迹')

---
## ✏️ 练习 1：实现 agent 循环

不看上面，自己把 `run_loop(llm, tools, task, max_steps)` 写出来。要点：维护 history、追加模型决策、`end_turn` 停止、分发工具并回灌观察、超 max_steps 返回 `status='max_steps'`。返回 `{'status', 'answer'(若 done), 'steps'}`。

In [ ]:
def run_loop(llm, tools, task, max_steps=10):
    history = [{'role':'user','content':task}]
    # TODO: for step in range(max_steps):
    #   resp = llm.complete(history, tools)
    #   把 resp 追加进 history (assistant 决策)
    #   若 resp['stop_reason']=='end_turn': 返回 {'status':'done','answer':resp['text'],'steps':step+1}
    #   否则对每个 resp['tool_calls'] 用 dispatch 执行, 把观察以 user 消息追加回 history
    # 循环结束仍未停 -> 返回 {'status':'max_steps','steps':max_steps}
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
tA = {'search': lambda q: f'结果({q})'}
bA = MockLLM([
    {'stop_reason':'tool_use','text':'','tool_calls':[{'id':'1','name':'search','input':{'q':'k'}}]},
    {'stop_reason':'end_turn','text':'完成'},
])
r = run_loop(bA, tA, '任务')
assert r['status'] == 'done' and r['answer'] == '完成' and r['steps'] == 2
# 死循环必须被 max_steps 拦住
loop = MockLLM([{'stop_reason':'tool_use','text':'','tool_calls':[{'id':'1','name':'search','input':{'q':'k'}}]}]*20)
r2 = run_loop(loop, tA, '任务', max_steps=3)
assert r2['status'] == 'max_steps' and r2['steps'] == 3
print('✅ 练习 1 通过：循环能正常完成、也能被 max_steps 拦住死循环')

## ✏️ 练习 2：健壮的 ReAct 解析

扩展 `parse_react_safe(text)`，把坏输入变成**可反馈的错误**而非崩溃：
(a) 有 `Final Answer:` -> `{'type':'final','text':...}`；
(b) 有合法 `Action:`+`Action Input:`(合法 JSON) -> `{'type':'tool','name','input'}`；
(c) 缺 Action / JSON 非法 / 两者都没 -> `{'type':'error','message':...}`（不抛异常）。

In [ ]:
def parse_react_safe(text):
    # TODO: 按 (a)(b)(c) 实现。(c) 用 try/except 把 json.loads 失败、缺字段都
    #       捕获成 {'type':'error','message': 原因}，绝不让异常冒出去。
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
good_tool = 'Action: calc\nAction Input: {"x": 1}'
good_final = 'Final Answer: 42'
bad_json = 'Action: calc\nAction Input: {x: 1}'      # 非法 JSON
no_action = 'Thought: 我在想但没决定'                  # 啥也没有
assert parse_react_safe(good_tool) == {'type':'tool','name':'calc','input':{'x':1}}
assert parse_react_safe(good_final) == {'type':'final','text':'42'}
assert parse_react_safe(bad_json)['type'] == 'error'   # 不崩，返回 error
assert parse_react_safe(no_action)['type'] == 'error'
print('✅ 练习 2 通过：坏输入都变成可反馈的 error，而非抛异常崩掉 agent')

## ✏️ 练习 3：终止状态机

实现 `classify_stop(resp, step, max_steps, cost, max_cost)`：根据一轮的情况返回终止判定。
返回 `'budget'`(cost>max_cost) / `'done'`(stop_reason=='end_turn') / `'continue'`(还要调工具) / `'max_steps'`(step 已是最后一步且要继续)。
优先级：预算 > 完成 > 步数。

In [ ]:
def classify_stop(resp, step, max_steps, cost, max_cost):
    # TODO: 依优先级返回 'budget'/'done'/'continue'/'max_steps'
    #   1) cost > max_cost -> 'budget'
    #   2) resp['stop_reason']=='end_turn' -> 'done'
    #   3) 否则要继续：若 step >= max_steps-1 -> 'max_steps' 否则 'continue'
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
tool_resp = {'stop_reason':'tool_use'}
final_resp = {'stop_reason':'end_turn'}
assert classify_stop(final_resp, 0, 10, 0.1, 1.0) == 'done'
assert classify_stop(tool_resp, 0, 10, 0.1, 1.0) == 'continue'
assert classify_stop(tool_resp, 9, 10, 0.1, 1.0) == 'max_steps'   # 最后一步还要继续
assert classify_stop(tool_resp, 0, 10, 2.0, 1.0) == 'budget'       # 预算优先
assert classify_stop(final_resp, 9, 10, 2.0, 1.0) == 'budget'      # 预算 > 完成
print('✅ 练习 3 通过：终止判定按 预算>完成>步数 的优先级正确')

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def run_loop(llm, tools, task, max_steps=10):
    history = [{'role':'user','content':task}]
    for step in range(max_steps):
        resp = llm.complete(history, tools)
        history.append({'role':'assistant','content':resp.get('text',''),
                        'tool_calls':resp.get('tool_calls')})
        if resp['stop_reason'] == 'end_turn':
            return {'status':'done','answer':resp['text'],'steps':step+1}
        for call in resp['tool_calls']:
            history.append({'role':'user','content':[dispatch(tools, call)]})
    return {'status':'max_steps','steps':max_steps}

In [ ]:
# 练习 2 参考答案
def parse_react_safe(text):
    if 'Final Answer:' in text:
        return {'type':'final','text':text.split('Final Answer:')[1].strip()}
    if 'Action:' not in text or 'Action Input:' not in text:
        return {'type':'error','message':'缺少 Action 或 Action Input'}
    try:
        name = text.split('Action:')[1].split('\n')[0].strip()
        raw = text.split('Action Input:')[1].strip()
        args = json.loads(raw)
        return {'type':'tool','name':name,'input':args}
    except (json.JSONDecodeError, IndexError) as e:
        return {'type':'error','message':f'解析失败: {e}'}

In [ ]:
# 练习 3 参考答案
def classify_stop(resp, step, max_steps, cost, max_cost):
    if cost > max_cost:
        return 'budget'
    if resp['stop_reason'] == 'end_turn':
        return 'done'
    return 'max_steps' if step >= max_steps - 1 else 'continue'

---
## 🧪 真实数据胶囊：把这个循环接到真实 Claude

现在把你写的 agent 循环接到**真实的 `claude-opus-4-8`**。关键：写一个与 MockLLM **同接口**的 `AnthropicLLM`，再用 `make_llm()` 做到 **有 key 接真模型、没 key 自动回退 MockLLM**——循环代码一个字都不用改。

> 下面的代码**无 key 也能跑**（自动回退 MockLLM）；有 `ANTHROPIC_API_KEY` 且装了 `anthropic` 时会真正调用 Claude。

In [ ]:
import os

class AnthropicLLM:
    '''与 MockLLM 同接口(complete)，内部调真实 Claude Messages API。
       把 Claude 的响应翻译成本课统一格式(stop_reason / tool_calls / text)。'''
    def __init__(self, model='claude-opus-4-8'):
        import anthropic                       # 仅真要用时才依赖
        self.client = anthropic.Anthropic()    # 读 ANTHROPIC_API_KEY
        self.model = model
    def complete(self, history, tools=None):
        # tools: 本课的工具 schema 列表(模块 02/03 详述)
        resp = self.client.messages.create(
            model=self.model, max_tokens=1024,
            messages=self._to_messages(history),
            tools=tools or [],
        )
        return self._to_decision(resp)
    def _to_messages(self, history):
        # 把本课 history 翻译成 Messages API 的 messages(此处省略细节, 模块 03 完整实现)
        return [m for m in history if m['role'] in ('user','assistant')]
    def _to_decision(self, resp):
        # stop_reason=='tool_use' -> 收集 tool_use 块; 否则取 text(end_turn)
        tool_calls, text = [], ''
        for blk in resp.content:
            if blk.type == 'text':
                text += blk.text
            elif blk.type == 'tool_use':
                tool_calls.append({'id':blk.id, 'name':blk.name, 'input':blk.input})
        return {'stop_reason':resp.stop_reason, 'text':text, 'tool_calls':tool_calls}

print('AnthropicLLM 定义完成(真实调用需 key + anthropic SDK)')

**🧪 胶囊练习**：实现 `make_llm(mock_script, model='claude-opus-4-8')`：有 `ANTHROPIC_API_KEY` 且能 import anthropic 就返回 `AnthropicLLM`，否则返回 `MockLLM(mock_script)`。**绝不能因为缺 key 抛异常**——这是本课的硬纪律。

In [ ]:
def make_llm(mock_script, model='claude-opus-4-8'):
    # TODO: 有 key 且能 import anthropic -> AnthropicLLM(model)
    #       否则(无 key 或没装 SDK) -> MockLLM(mock_script)，绝不抛异常
    raise NotImplementedError

In [ ]:
# 自测：本机无论有无 key 都应得到一个可用的 llm，且不报错
llm = make_llm([{'stop_reason':'end_turn','text':'hi'}])
assert hasattr(llm, 'complete'), 'make_llm 必须返回带 complete 的对象'
# 无 key 时应回退到 MockLLM 并能端到端驱动循环
if not os.environ.get('ANTHROPIC_API_KEY'):
    assert isinstance(llm, MockLLM), '无 key 应回退 MockLLM'
    out = run_agent_full(llm, {}, '你好')
    assert out['status'] == 'done'
print('✅ 胶囊练习通过：有 key 接真 Claude、没 key 自动回退 MockLLM，循环代码不变')

In [ ]:
# 📖 胶囊参考答案
def make_llm(mock_script, model='claude-opus-4-8'):
    if os.environ.get('ANTHROPIC_API_KEY'):
        try:
            import anthropic  # noqa: F401
            return AnthropicLLM(model)
        except ImportError:
            pass
    return MockLLM(mock_script)

### 小结
- agent 循环 = `while`：**感知→决策→行动→观察**，反复『问模型→执行→把结果追加回 history』直到 `end_turn` 或触发上限。
- **ReAct**：让模型交替写 Thought/Action/Observation；解析是最易出 bug 的一步，坏输入要变成**可反馈的错误**而非崩溃。
- **停止**：正常(`end_turn`) + 强制(max_steps / 预算 / 死循环)；每次退出带明确终止状态。
- **状态**：记忆全在 history；**忘了追加模型决策 = 失忆 = 死循环**。
- **接真实 Claude**：同接口 `AnthropicLLM` + `make_llm()` 自动回退，循环代码不变。

下一站：**模块 02 · 工具系统** —— 把循环里那个 `dispatch` 做扎实：注册、schema、分发、并行、错误隔离。